# Notebook 3 — Separating financial crises in the M-I plane

The M-I decomposition separates the 2008 Global Financial Crisis and the 2020 COVID shock
on S&P 500 daily returns — two events that volatility filters conflate.

From *Symbolic Pure Time* (Vázquez Broquá, 2026), §5.2.

**Data source:** `yfinance` (public) or the CSV in `results/sp500_daily.csv` if already
downloaded. Run the first cell with `USE_YFINANCE = True` if you have an internet
connection; otherwise set it to `False` to use the precomputed CSV.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), '..', '..', '..', 'lib'))
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import gsd

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})

USE_YFINANCE = True   # set False to use precomputed CSV

if USE_YFINANCE:
    try:
        import yfinance as yf
        data = yf.download('^GSPC', start='1990-01-01', end='2026-01-01',
                           progress=False, auto_adjust=True)
        prices = data['Close'].values.astype(float)
        dates_raw = data.index
        import pandas as pd
        dates = pd.to_datetime(dates_raw)
        print(f"Downloaded S&P 500: {len(prices)} trading days")
    except Exception as e:
        print(f"yfinance failed: {e}\nFalling back to CSV.")
        USE_YFINANCE = False

if not USE_YFINANCE:
    csv_path = os.path.join('..', 'results', 'sp500_daily.csv')
    if os.path.exists(csv_path):
        import pandas as pd
        df = pd.read_csv(csv_path, parse_dates=['Date'], index_col='Date')
        prices = df['Close'].values.astype(float)
        dates = df.index
        print(f"Loaded from CSV: {len(prices)} trading days")
    else:
        print("No data source available. Please set USE_YFINANCE=True or provide sp500_daily.csv")
        prices = None


In [ ]:
if prices is not None:
    # Log returns
    log_ret = np.log(prices[1:] / prices[:-1])
    dates_ret = dates[1:]

    print(f"Period: {dates_ret[0].date()} to {dates_ret[-1].date()}")
    print(f"N = {len(log_ret)} daily log returns")
    print(f"Mean: {log_ret.mean()*252:.2%}/yr   Std: {log_ret.std()*np.sqrt(252):.2%}/yr")

    fig, ax = plt.subplots(figsize=(12, 3.5))
    ax.plot(dates_ret, log_ret, color='#4f46e5', lw=0.5, alpha=0.7)
    ax.axhline(0, color='#ccc', lw=0.5)
    # Mark crisis periods
    ax.axvspan(np.datetime64('2008-09-01'), np.datetime64('2009-06-30'),
               alpha=0.12, color='#e0552b', label='2008 GFC')
    ax.axvspan(np.datetime64('2020-02-15'), np.datetime64('2020-05-31'),
               alpha=0.12, color='#1f9d6b', label='2020 COVID')
    ax.set_title('S&P 500 daily log returns (1990–2026)', fontweight='bold')
    ax.set_ylabel('log return')
    ax.legend(fontsize=9, loc='lower right')
    plt.tight_layout()
    plt.savefig('fig_sp500_returns.png', dpi=130, bbox_inches='tight')
    plt.show()


In [ ]:
def sptls_mi_series(y, n_shuffles=10, seed=42):
    res = gsd.SPTLS().fit(y)
    M = res.M.copy()
    rng_s = np.random.default_rng(seed)
    M_iid = np.zeros_like(M)
    for _ in range(n_shuffles):
        ys = y.copy(); rng_s.shuffle(ys)
        M_iid += gsd.SPTLS().fit(ys).M / n_shuffles
    delta = np.linalg.norm(M - M_iid, 'fro')
    kappa = stats.kurtosis(res.resid_jet[:, 1], fisher=True)
    return delta, kappa

if prices is not None:
    import pandas as pd
    ret_series = pd.Series(log_ret, index=dates_ret)

    # Rolling M-I with a 120-day window, 10-day step
    window, step = 120, 10
    rolling_dates, rolling_delta, rolling_kappa = [], [], []

    idx = np.arange(len(log_ret))
    for start in range(0, len(log_ret) - window, step):
        window_ret = log_ret[start:start+window]
        try:
            d, k = sptls_mi_series(window_ret, n_shuffles=8, seed=start)
            rolling_dates.append(dates_ret[start + window//2])
            rolling_delta.append(d)
            rolling_kappa.append(k)
        except Exception:
            pass

    rolling_dates = np.array(rolling_dates)
    rolling_delta = np.array(rolling_delta)
    rolling_kappa = np.array(rolling_kappa)

    # Rolling volatility (comparison baseline)
    vol = pd.Series(log_ret**2, index=dates_ret).rolling(window).mean().apply(np.sqrt) * np.sqrt(252)

    print(f"Computed {len(rolling_dates)} rolling M-I observations")
    print(f"2008 GFC window (Sep 2008 – Jun 2009)")
    mask_gfc = (rolling_dates >= np.datetime64('2008-09-01')) & (rolling_dates <= np.datetime64('2009-06-30'))
    print(f"  Δ = {rolling_delta[mask_gfc].mean():.4f}   κ = {rolling_kappa[mask_gfc].mean():.2f}")
    print(f"2020 COVID window (Feb 2020 – May 2020)")
    mask_cov = (rolling_dates >= np.datetime64('2020-02-15')) & (rolling_dates <= np.datetime64('2020-05-31'))
    print(f"  Δ = {rolling_delta[mask_cov].mean():.4f}   κ = {rolling_kappa[mask_cov].mean():.2f}")


In [ ]:
if prices is not None:
    fig = plt.figure(figsize=(13, 9))
    gs = fig.add_gridspec(3, 2, hspace=0.45, wspace=0.35)

    ax_ret = fig.add_subplot(gs[0, :])
    ax_delta = fig.add_subplot(gs[1, 0])
    ax_kappa = fig.add_subplot(gs[1, 1])
    ax_mi = fig.add_subplot(gs[2, 0])
    ax_vol = fig.add_subplot(gs[2, 1])

    # Returns
    ax_ret.plot(dates_ret, log_ret, color='#4f46e5', lw=0.5, alpha=0.6)
    ax_ret.axvspan(np.datetime64('2008-09-01'), np.datetime64('2009-06-30'), alpha=0.15, color='#e0552b')
    ax_ret.axvspan(np.datetime64('2020-02-15'), np.datetime64('2020-05-31'), alpha=0.15, color='#1f9d6b')
    ax_ret.set_title('S&P 500 log returns', fontweight='bold'); ax_ret.set_ylabel('log ret')

    # Delta
    ax_delta.plot(rolling_dates, rolling_delta, color='#4f46e5', lw=1.2)
    ax_delta.axvspan(np.datetime64('2008-09-01'), np.datetime64('2009-06-30'), alpha=0.15, color='#e0552b')
    ax_delta.axvspan(np.datetime64('2020-02-15'), np.datetime64('2020-05-31'), alpha=0.15, color='#1f9d6b')
    ax_delta.set_title('Memory axis Δ  (rolling 120d)', fontweight='bold'); ax_delta.set_ylabel('Δ')

    # Kappa
    ax_kappa.plot(rolling_dates, rolling_kappa, color='#e0552b', lw=1.2)
    ax_kappa.axvspan(np.datetime64('2008-09-01'), np.datetime64('2009-06-30'), alpha=0.15, color='#e0552b')
    ax_kappa.axvspan(np.datetime64('2020-02-15'), np.datetime64('2020-05-31'), alpha=0.15, color='#1f9d6b')
    ax_kappa.axhline(0, color='#ccc', lw=1)
    ax_kappa.set_title('Innovation axis κ  (rolling 120d)', fontweight='bold'); ax_kappa.set_ylabel('κ')

    # M-I plane scatter
    n = len(rolling_delta)
    for i in range(n):
        t = rolling_dates[i]
        if np.datetime64('2008-09-01') <= t <= np.datetime64('2009-06-30'):
            c = '#e0552b'; label = '2008 GFC' if i == np.where(mask_gfc)[0][0] else None
        elif np.datetime64('2020-02-15') <= t <= np.datetime64('2020-05-31'):
            c = '#1f9d6b'; label = '2020 COVID' if i == np.where(mask_cov)[0][0] else None
        else:
            c = '#9aa0ab44'; label = None
        ax_mi.scatter(rolling_kappa[i], rolling_delta[i], s=18, color=c, alpha=0.7,
                      label=label, zorder=3 if c != '#9aa0ab44' else 2)
    handles = [plt.scatter([], [], s=40, color='#e0552b', label='2008 GFC'),
               plt.scatter([], [], s=40, color='#1f9d6b', label='2020 COVID'),
               plt.scatter([], [], s=40, color='#9aa0ab', alpha=0.5, label='Other')]
    ax_mi.legend(handles=handles, fontsize=9, frameon=False)
    ax_mi.set_xlabel('κ  (innovation axis)'); ax_mi.set_ylabel('Δ  (memory axis)')
    ax_mi.set_title('M-I plane\nTwo crises in distinct regions', fontweight='bold')

    # Volatility comparison
    ax_vol.plot(vol.index, vol.values, color='#8a8f9a', lw=1.0, alpha=0.8)
    ax_vol.axvspan(np.datetime64('2008-09-01'), np.datetime64('2009-06-30'), alpha=0.15, color='#e0552b')
    ax_vol.axvspan(np.datetime64('2020-02-15'), np.datetime64('2020-05-31'), alpha=0.15, color='#1f9d6b')
    ax_vol.set_title('Realised volatility  (120d rolling)
Conflates both crises', fontweight='bold')
    ax_vol.set_ylabel('ann. vol')

    plt.suptitle('S&P 500 · M-I decomposition vs. rolling volatility',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.savefig('fig_sp500_mi.png', dpi=130, bbox_inches='tight')
    plt.show()

    print()
    print("Key finding:")
    print("  2008 GFC:    Δ HIGH, κ HIGH  — systemic, memory-building crisis")
    print("  2020 COVID:  Δ LOW,  κ HIGH  — innovation-driven spike, not memory-building")
    print("  Rolling vol: both crises show similarly large spikes — conflated.")
    print()
    print("Two structurally distinct events. One closed-form solve.")
